# **Installation and Licensing**

In [ ]:
!pip install -q gamspy # Installs the GAMSPy package. GAMSPy documentation: https://gamspy.readthedocs.io/en/latest/

In [ ]:
!gamspy show license

In [ ]:
!gamspy list solvers

In [ ]:
!gamspy install license <path_to_ascii_file or access code> # paste your license here (if demo-license is not enough)

In [ ]:
# Optional: Install all solvers
!gamspy install solver --install-all-solvers

In [ ]:
!gamspy list solvers # Lists all available solvers in the gamspy package

# **Mathematical Model**

**Cardinality-Constrained Portfolio Optimization with Transaction Costs and Risk Limit**

\begin{align}
    \max \quad & \sum_{i=1}^n \mu_i x_i - \sum_{i=1}^n c_i (x_i - x_i^{prev})^2 \\
    \text{subject to} \quad
    & \sum_{i=1}^n x_i = 1 \quad && \text{(Budget constraint)} \\
    & x_i \leq y_i, \quad \forall i = 1, \ldots, n \quad && \text{(Linking constraint)} \\
    & \sum_{i=1}^n y_i \leq 5 \quad && \text{(Cardinality constraint)} \\
    & \sum_{i=1}^n \sum_{j=1}^n Q_{ij} x_i x_j \leq \sigma^2_{\text{max}} \quad && \text{(Risk constraint)} \\
    & x_i \geq 0, \quad \forall i = 1, \ldots, n \quad && \text{(Non-negativity)} \\
    & y_i \in \{0, 1\}, \quad \forall i = 1, \ldots, n \quad && \text{(Binary selection variables)}
\end{align}

$$
\begin{array}{l l l r r r r}
\text{Name} & \text{Type} & \text{C} & \#\text{Vars} & \#\text{BinVars} & \#\text{IntVars} & \#\text{Cons} \\
\hline
CCPOPwTCaRL5\_20 & MBQCQP & Convex & 40 & 20 & 0 & 23 \\
CCPOPwTCaRL5\_50 & MBQCQP & Convex & 100 & 50 & 0 & 53 \\
CCPOPwTCaRL5\_165 & MBQCQP & Convex & 330 & 165 & 0  & 168 \\
CCPOPwTCaRL5\_308 & MBQCQP & Convex & 616 & 308 & 0  & 311 \\
CCPOPwTCaRL5\_538 & MBQCQP & Convex & 1076 & 538  & 0  & 541  \\
\end{array}
$$


#  **Solver Setup**


## CCPOPwTCaRL5\_20

In [ ]:
import yfinance as yf
import numpy as np
import pandas as pd
from gamspy import Container, Set, Parameter, Variable, Equation, Model, Sum, Sense, Alias

# Download market data
tickers = [
    "BA", "CAT", "CVX", "DIS", "JNJ", "KO", "MCD", "MRK", "NKE", "PFE",
    "CRM", "IBM", "INTU", "LOW", "MDT", "MMM", "RTX", "TMO", "UNP", "ZTS"
]
data = yf.download(tickers, start='2022-01-01', end='2025-05-31')['Close']
returns = data.pct_change().dropna()

# Expected returns (mean of returns)
mu_array = returns.mean().values
cov = returns.cov().values
n_assets = len(cov)
assets = [f"asset_{i+1}" for i in range(n_assets)]

# Parameters for transaction cost
x_prev_array = np.array([1/n_assets] * n_assets)  # equally weighted previous portfolio
c_array = np.random.uniform(0.001, 0.01, size=n_assets)  # transaction cost coefficients

# Risk budget parameter (max acceptable variance)
sigma_max_sq = 0.01  # Adjust this risk tolerance as you see fit

# Create GAMSPy container
m = Container()

# Sets
i = Set(m, name="i", records=assets)
ii = Alias(m, name="ii", alias_with=i)

# Parameters
Q = Parameter(m, name="Q", domain=[i, ii],
              records=[(assets[r], assets[c], cov[r, c]) for r in range(n_assets) for c in range(n_assets)])

mu = Parameter(m, name="mu", domain=[i],
               records=[(assets[k], mu_array[k]) for k in range(n_assets)])

x_prev = Parameter(m, name="x_prev", domain=[i],
                   records=[(assets[k], x_prev_array[k]) for k in range(n_assets)])

c = Parameter(m, name="c", domain=[i],
              records=[(assets[k], c_array[k]) for k in range(n_assets)])

# Scalar parameter for max risk (variance)
sigma_max = sigma_max_sq  # just use a plain Python scalar

# Variables
x = Variable(m, name="x", domain=[i], type="Free")  # portfolio weights (can be zero or positive)
x.lo[i] = 0  # no short selling

y = Variable(m, name="y", domain=[i], type="Binary")  # selection binary variables

# Constraints
budget = Equation(m, name="budget")
budget[...] = Sum(i, x[i]) == 1

link = Equation(m, name="link", domain=[i])
link[i] = x[i] <= y[i]  # link between selection and investment

cardinality = Equation(m, name="cardinality")
cardinality[...] = Sum(i, y[i]) <= 5  # max 5 assets selected

risk = Equation(m, name="risk")
risk[...] = Sum([i, ii], Q[i, ii] * x[i] * x[ii]) <= sigma_max

# Objective: maximize expected return minus transaction cost (transaction cost is quadratic)
transaction_cost = Sum(i, c[i] * (x[i] - x_prev[i]) * (x[i] - x_prev[i]))
objective_expr = Sum(i, mu[i] * x[i]) - transaction_cost

# Model
model = Model(m, name="CCPOPwTCaRL5_20",
              equations=m.getEquations(),
              problem="MIQCP",
              sense=Sense.MAX,
              objective=objective_expr)

# Solve
import sys
model.solve(output=sys.stdout, solver="SHOT", solver_options={"Dual.MIP.Solver": 2})

In [ ]:
# Convert GAMSPy model to GAMS (Note: One needs to download both the .gms and .gdx file)
model.toGams(path="gams")

After this, the files were opened and combined in GAMS Studio by GAMS Convert. Below is the rest of the codes for this type of problem.


## CCPOPwTCaRL5\_50

In [ ]:
import yfinance as yf
import numpy as np
import pandas as pd
from gamspy import Container, Set, Parameter, Variable, Equation, Model, Sum, Sense, Alias

# Download market data
tickers = [
    "BA", "CAT", "CVX", "DIS", "JNJ", "KO", "MCD", "MRK", "NKE", "PFE",
    "CRM", "IBM", "INTU", "LOW", "MDT", "MMM", "RTX", "TMO", "UNP", "ZTS",
    "BDX", "ICE", "CL", "ITW", "APD", "SHW", "EOG", "TRV", "FDX", "HUM",
    "NKE", "PSA", "EMR", "ETN", "TFC", "GM", "AON", "WM", "ROP", "AIG",
    "MAR", "USB", "MCO", "D", "EW", "IDXX", "AEP", "FTNT", "ADSK", "SPG", "WTW"
]
data = yf.download(tickers, start='2022-01-01', end='2025-05-31')['Close']
returns = data.pct_change().dropna()

# Expected returns (mean of returns)
mu_array = returns.mean().values
cov = returns.cov().values
n_assets = len(cov)
assets = [f"asset_{i+1}" for i in range(n_assets)]

# Parameters for transaction cost
x_prev_array = np.array([1/n_assets] * n_assets)  # equally weighted previous portfolio
c_array = np.random.uniform(0.001, 0.01, size=n_assets)  # transaction cost coefficients

# Risk budget parameter (max acceptable variance)
sigma_max_sq = 0.01  # Adjust this risk tolerance as you see fit

# Create GAMSPy container
m = Container()

# Sets
i = Set(m, name="i", records=assets)
ii = Alias(m, name="ii", alias_with=i)

# Parameters
Q = Parameter(m, name="Q", domain=[i, ii],
              records=[(assets[r], assets[c], cov[r, c]) for r in range(n_assets) for c in range(n_assets)])

mu = Parameter(m, name="mu", domain=[i],
               records=[(assets[k], mu_array[k]) for k in range(n_assets)])

x_prev = Parameter(m, name="x_prev", domain=[i],
                   records=[(assets[k], x_prev_array[k]) for k in range(n_assets)])

c = Parameter(m, name="c", domain=[i],
              records=[(assets[k], c_array[k]) for k in range(n_assets)])

# Scalar parameter for max risk (variance)
sigma_max = sigma_max_sq  # just use a plain Python scalar

# Variables
x = Variable(m, name="x", domain=[i], type="Free")  # portfolio weights (can be zero or positive)
x.lo[i] = 0  # no short selling

y = Variable(m, name="y", domain=[i], type="Binary")  # selection binary variables

# Constraints
budget = Equation(m, name="budget")
budget[...] = Sum(i, x[i]) == 1

link = Equation(m, name="link", domain=[i])
link[i] = x[i] <= y[i]  # link between selection and investment

cardinality = Equation(m, name="cardinality")
cardinality[...] = Sum(i, y[i]) <= 5  # max 5 assets selected

risk = Equation(m, name="risk")
risk[...] = Sum([i, ii], Q[i, ii] * x[i] * x[ii]) <= sigma_max

# Objective: maximize expected return minus transaction cost (transaction cost is quadratic)
transaction_cost = Sum(i, c[i] * (x[i] - x_prev[i]) * (x[i] - x_prev[i]))
objective_expr = Sum(i, mu[i] * x[i]) - transaction_cost

# Model
model = Model(m, name="CCPOPwTCaRL5_50",
              equations=m.getEquations(),
              problem="MIQCP",
              sense=Sense.MAX,
              objective=objective_expr)

# Solve
import sys
model.solve(output=sys.stdout, solver="SHOT", solver_options={"Dual.MIP.Solver": 2})

## CCPOPwTCaRL5\_165


In [ ]:
import yfinance as yf
import numpy as np
import pandas as pd
from gamspy import Container, Set, Parameter, Variable, Equation, Model, Sum, Sense, Alias

# Download market data
tickers = [
    "XOM", "UNH", "LLY", "JNJ", "PG", "MA", "HD", "MRK", "CVX",
    "PEP", "ABBV", "COST", "BAC", "ADBE", "KO", "WMT", "CRM", "NFLX", "TMO",
    "ABT", "ACN", "PFE", "CSCO", "MCD", "DHR", "LIN", "INTC", "QCOM", "NEE",
    "WFC", "TXN", "PM", "MS", "BMY", "AMGN", "HON", "ORCL", "UNP", "AVB",
    "LOW", "UPS", "RTX", "AMD", "GS", "SPGI", "CVS", "SBUX", "BLK", "MDT",
    "ISRG", "LMT", "INTU", "GE", "CAT", "TJX", "AMAT", "AXP", "DE", "NOW",
    "ADI", "PLD", "ZTS", "SYK", "BA", "MO", "CI", "TGT", "BKNG", "GILD",
    "MMC", "ADP", "VRTX", "CB", "ELV", "REGN", "PNC", "FI", "C", "ADI",
    "CSX", "MDLZ", "SCHW", "PGR", "MU", "DUK", "EQIX", "SO", "HCA",
    "DVN", "SBAC", "OTIS", "EFX", "CHTR", "BKR", "DFS", "KEYS", "WMB", "VRSK",
    "FTV", "STT", "NEM", "AME", "PPG", "CEG", "NOC", "ES", "ANET", "CTVA",
    "PAYX", "BIIB", "XEL", "LHX", "MTD", "DRI", "ZBH", "KLAC", "TSCO", "TDG",
    "RMD", "PCG", "DOV", "VICI", "CBRE", "GWW", "CMS", "EXC", "DLTR",
    "ATO", "CDW", "HIG", "LDOS", "ALGN", "CNP", "MLM", "LRCX", "AEE", "EIX",
    "RCL", "FANG", "PPL", "FAST", "CARR", "IFF", "COR", "ETR", "TRGP", "MAA",
    "AKAM", "TSN", "EPAM", "NUE", "WEC", "HPE", "IR", "AIZ", "MKC", "RSG",
    "CAH", "VTR", "CHD", "BALL", "NDAQ", "NI", "AVY", "CRL", "DTE",
]
data = yf.download(tickers, start='2022-01-01', end='2025-05-31')['Close']
returns = data.pct_change().dropna()

# Expected returns (mean of returns)
mu_array = returns.mean().values
cov = returns.cov().values
n_assets = len(cov)
assets = [f"asset_{i+1}" for i in range(n_assets)]

# Parameters for transaction cost
x_prev_array = np.array([1/n_assets] * n_assets)  # equally weighted previous portfolio
c_array = np.random.uniform(0.001, 0.01, size=n_assets)  # transaction cost coefficients

# Risk budget parameter (max acceptable variance)
sigma_max_sq = 0.01  # Adjust this risk tolerance as you see fit

# Create GAMSPy container
m = Container()

# Sets
i = Set(m, name="i", records=assets)
ii = Alias(m, name="ii", alias_with=i)

# Parameters
Q = Parameter(m, name="Q", domain=[i, ii],
              records=[(assets[r], assets[c], cov[r, c]) for r in range(n_assets) for c in range(n_assets)])

mu = Parameter(m, name="mu", domain=[i],
               records=[(assets[k], mu_array[k]) for k in range(n_assets)])

x_prev = Parameter(m, name="x_prev", domain=[i],
                   records=[(assets[k], x_prev_array[k]) for k in range(n_assets)])

c = Parameter(m, name="c", domain=[i],
              records=[(assets[k], c_array[k]) for k in range(n_assets)])

# Scalar parameter for max risk (variance)
sigma_max = sigma_max_sq  # just use a plain Python scalar

# Variables
x = Variable(m, name="x", domain=[i], type="Free")  # portfolio weights (can be zero or positive)
x.lo[i] = 0  # no short selling

y = Variable(m, name="y", domain=[i], type="Binary")  # selection binary variables

# Constraints
budget = Equation(m, name="budget")
budget[...] = Sum(i, x[i]) == 1

link = Equation(m, name="link", domain=[i])
link[i] = x[i] <= y[i]  # link between selection and investment

cardinality = Equation(m, name="cardinality")
cardinality[...] = Sum(i, y[i]) <= 5  # max 5 assets selected

risk = Equation(m, name="risk")
risk[...] = Sum([i, ii], Q[i, ii] * x[i] * x[ii]) <= sigma_max

# Objective: maximize expected return minus transaction cost (transaction cost is quadratic)
transaction_cost = Sum(i, c[i] * (x[i] - x_prev[i]) * (x[i] - x_prev[i]))
objective_expr = Sum(i, mu[i] * x[i]) - transaction_cost

# Model
model = Model(m, name="CCPOPwTCaRL5_165",
              equations=m.getEquations(),
              problem="MIQCP",
              sense=Sense.MAX,
              objective=objective_expr)

# Solve
import sys
model.solve(output=sys.stdout, solver="SHOT", solver_options={"Dual.MIP.Solver": 2})


## CCPOPwTCaRL5\_308


In [ ]:
import yfinance as yf
import numpy as np
import pandas as pd
from gamspy import Container, Set, Parameter, Variable, Equation, Model, Sum, Sense, Alias

# Download market data
tickers = [
    "ADBE", "CMCSA", "PEP", "NFLX", "AMD", "GILD",
    "BA", "CAT", "CVX", "DIS", "JNJ", "KO", "MCD", "MRK", "NKE", "PFE",
    "PG", "T", "VZ", "WMT", "XOM", "GS", "JPM", "UNH", "HD", "ORCL",
    "CRM", "IBM", "INTU", "LOW", "MDT", "MMM", "RTX", "TMO", "UNP", "ZTS",
    "AMGN", "BLK", "EL", "FDX", "GE", "GM",
    "HON", "IBM", "ISRG", "JNJ", "LMT", "LIN", "LRCX", "MAR", "MDLZ", "MET",
    "MO", "MS", "NEE", "NOC", "NVDA", "PYPL", "SCHW", "SO", "SPGI",
    "TGT", "TMUS", "TXN", "UPS", "USB", "V", "VLO", "VZ", "WBA", "WFC",
    "XEL", "ZBRA", "ZBH", "SNAP", "DOCU", "UBER", "ZM", "SHOP",
     "ADP", "BMY", "C",  "DOW",  "EMR",  "F",
    "GM",  "HCA",   "LVS",  "PYPL", "BK",
    "AAPL", "MSFT", "AMZN", "NVDA", "GOOGL", "META", "AVGO", "JPM", "TSLA",
    "XOM", "UNH", "LLY", "JNJ", "PG", "MA", "HD", "MRK", "CVX",
    "PEP", "ABBV", "COST", "BAC", "ADBE", "KO", "WMT", "CRM", "NFLX", "TMO",
    "ABT", "ACN", "PFE", "CSCO", "INTC", "QCOM", "NEE",
    "WFC", "TXN", "PM", "MS", "BMY", "AMGN", "HON", "ORCL", "UNP", "AVB",
    "LOW", "UPS", "RTX", "AMD", "GS", "SPGI", "CVS", "BLK", "MDT",
    "ISRG", "LMT", "INTU", "GE", "CAT", "TJX", "AMAT", "AXP", "DE", "NOW",
    "ADI", "PLD", "ZTS", "BKNG", "GILD",
    "MMC", "ADP", "VRTX", "CB", "ELV", "REGN", "PNC", "FI", "C", "ADI",
    "CSX", "MDLZ", "SCHW", "PGR", "MU", "DUK", "EQIX", "SO", "HCA",
    "BDX", "ICE", "CL", "ITW", "APD", "SHW", "EOG", "TRV", "FDX", "HUM",
    "NKE", "PSA", "EMR", "ETN", "ROP", "AIG",
    "MAR", "USB", "D", "EW", "IDXX", "AEP", "FTNT", "ADSK",
    "ROST", "MNST", "KDP", "CMG", "MET", "CTAS", "ORLY", "PCAR", "CTSH", "AZO",
    "SPG", "WTW", "CME", "DLR", "PH", "F", "VLO", "PRU", "OXY", "MSI",
    "ALL", "WELL", "TEL", "SRE", "HLT", "HSY", "PEG", "CDNS", "YUM", "HPQ",
    "EBAY", "DHI", "HES", "MCK", "COF", "KR", "WBD", "ED", "TT", "STZ",
    "DVN", "SBAC", "OTIS", "VRSK",
    "FTV", "STT", "NEM", "AME", "PPG", "CEG", "NOC", "ES", "ANET", "CTVA",
    "PAYX", "BIIB", "XEL", "LHX", "MTD", "DRI", "ZBH", "KLAC", "TSCO", "TDG",
    "RMD", "PCG", "DOV", "CMS", "EXC", "DLTR",
    "ATO", "CDW", "HIG", "LDOS", "ALGN", "CNP", "MLM", "LRCX", "AEE", "EIX",
    "RCL", "FANG", "PPL", "FAST", "CARR", "IFF", "MAA",
    "AKAM", "TSN", "EPAM", "NUE", "WEC", "HPE", "IR", "AIZ", "MKC", "RSG",
    "CAH", "VTR", "CRL", "DTE",
    "SJM", "FE", "WAT", "TTWO", "EXR", "BBY", "HOLX", "SWKS", "JKHY", "LKQ",
    "FMC", "TER", "NRG", "AES", "K", "TYL", "PHM", "JBHT",
    "RL", "BR", "ZBRA", "BXP", "GNRC", "VMC", "STE", "ESS", "APA",
    "INCY", "TECH", "PKG", "CF", "HBAN", "AFL", "NVR", "ALB", "TXT", "DXCM",
    "KIM", "UHS", "ARE", "PWR",
    "WHR", "LNT", "EMN", "TAP", "AMCR", "XRAY", "KMX", "L", "CINF", "SNA",
    "FRT", "NOV", "CAG", "GRMN", "WRB", "DXC", "OGN", "TPR", "PNW",
    "A", "SEE", "ALLE", "AOS",
    "LKQ", "IPG", "QRVO", "WY", "GNW", "TXT", "MTB", "ZION", "PFG"
] #

data = yf.download(tickers, start='2022-01-01', end='2025-05-31')['Close']
returns = data.pct_change().dropna()

# Expected returns (mean of returns)
mu_array = returns.mean().values
cov = returns.cov().values
n_assets = len(cov)
assets = [f"asset_{i+1}" for i in range(n_assets)]

# Parameters for transaction cost
x_prev_array = np.array([1/n_assets] * n_assets)  # equally weighted previous portfolio
c_array = np.random.uniform(0.001, 0.01, size=n_assets)  # transaction cost coefficients

# Risk budget parameter (max acceptable variance)
sigma_max_sq = 0.01  # Adjust this risk tolerance as you see fit

# Create GAMSPy container
m = Container()

# Sets
i = Set(m, name="i", records=assets)
ii = Alias(m, name="ii", alias_with=i)

# Parameters
Q = Parameter(m, name="Q", domain=[i, ii],
              records=[(assets[r], assets[c], cov[r, c]) for r in range(n_assets) for c in range(n_assets)])

mu = Parameter(m, name="mu", domain=[i],
               records=[(assets[k], mu_array[k]) for k in range(n_assets)])

x_prev = Parameter(m, name="x_prev", domain=[i],
                   records=[(assets[k], x_prev_array[k]) for k in range(n_assets)])

c = Parameter(m, name="c", domain=[i],
              records=[(assets[k], c_array[k]) for k in range(n_assets)])

# Scalar parameter for max risk (variance)
sigma_max = sigma_max_sq  # just use a plain Python scalar

# Variables
x = Variable(m, name="x", domain=[i], type="Free")  # portfolio weights (can be zero or positive)
x.lo[i] = 0  # no short selling

y = Variable(m, name="y", domain=[i], type="Binary")  # selection binary variables

# Constraints
budget = Equation(m, name="budget")
budget[...] = Sum(i, x[i]) == 1

link = Equation(m, name="link", domain=[i])
link[i] = x[i] <= y[i]  # link between selection and investment

cardinality = Equation(m, name="cardinality")
cardinality[...] = Sum(i, y[i]) <= 5  # max 5 assets selected

risk = Equation(m, name="risk")
risk[...] = Sum([i, ii], Q[i, ii] * x[i] * x[ii]) <= sigma_max

# Objective: maximize expected return minus transaction cost (transaction cost is quadratic)
transaction_cost = Sum(i, c[i] * (x[i] - x_prev[i]) * (x[i] - x_prev[i]))
objective_expr = Sum(i, mu[i] * x[i]) - transaction_cost

# Model
model = Model(m, name="CCPOPwTCaRL5_308",
              equations=m.getEquations(),
              problem="MIQCP",
              sense=Sense.MAX,
              objective=objective_expr)

# Solve
import sys
model.solve(output=sys.stdout, solver="SHOT", solver_options={"Dual.MIP.Solver": 2})

## CCPOPwTCaRL5\_538



In [ ]:
import yfinance as yf
import numpy as np
import pandas as pd
from gamspy import Container, Set, Parameter, Variable, Equation, Model, Sum, Sense, Alias

# Download market data

tickers = [
    "ADBE", "CMCSA", "PEP", "NFLX", "AMD", "GILD",
    "BA", "CAT", "CVX", "DIS", "JNJ", "KO", "MCD", "MRK", "NKE", "PFE",
     "T", "VZ", "WMT", "XOM", "GS", "JPM", "UNH", "HD", "ORCL",
    "CRM", "IBM", "INTU", "LOW", "MDT", "MMM", "RTX", "TMO", "UNP", "ZTS",
    "AMGN", "BLK", "EL", "FDX", "GE", "GM", "HON", "ISRG", "LMT", "LIN",
    "LRCX", "MAR", "MDLZ", "MET", "MO", "MS", "NEE", "NOC", "NVDA", "PYPL",
    "SCHW", "SO", "SPGI", "TGT", "TMUS", "TXN", "UPS", "USB", "V", "VLO",
    "WFC", "XEL", "ZBRA", "ZBH", "SNAP", "DOCU", "UBER", "ZM", "SHOP", "ADP",
    "ESS", "APA", "INCY", "TECH", "PKG", "CF", "HBAN", "AFL", "NVR", "ALB",
    "TXT", "DXCM", "KIM", "UHS", "ARE", "PWR", "WHR", "LNT", "EMN", "TAP",
    "AMCR", "XRAY", "KMX", "L", "CINF", "SNA", "FRT", "NOV", "CAG", "GRMN",
    "WRB", "DXC", "OGN", "TPR", "PNW", "A", "SEE", "ALLE", "AOS", "IPG",
   "QRVO", "WY", "GNW", "MTB", "ZION", "PFG", "AAL", "ALK", "AMP", "AMT",
    "APTV", "AVY", "BAX", "BEN", "BKR", "BWA", "CBRE", "CCI", "CHD", "CHRW",
    "CLX", "CMA", "CMI", "CNC", "COO", "COP", "CZR", "D", "DAL", "DD",
     "DG", "DGX", "DHR", "EA", "ECL", "EFX", "ETR", "EVRG",
    "EXPD", "EXPE"
]


#ticckers = sorted(set(tickers_sp500).union(tickers).union(tickers_nasdaq100))
ticckers = sorted(set(tickers_nasdaq100).union(tickers))
data = yf.download(
    ticckers,
    start="2023-01-01",
    end="2025-05-31",
    auto_adjust=True,
    progress=False
)["Close"]

# Remove tickers with no price data at all
data = data.dropna(axis=1, how="all")

# Remove tickers with partial histories (required for covariance stability)
data = data.dropna(axis=1, how="any")

returns = data.pct_change(fill_method=None).dropna()


# Expected returns (mean of returns)
mu_array = returns.mean().values
cov = returns.cov().values
n_assets = len(cov)
assets = [f"asset_{i+1}" for i in range(n_assets)]

# Parameters for transaction cost
x_prev_array = np.array([1/n_assets] * n_assets)  # equally weighted previous portfolio
c_array = np.random.uniform(0.001, 0.01, size=n_assets)  # transaction cost coefficients

# Risk budget parameter (max acceptable variance)
sigma_max_sq = 0.01  # Adjust this risk tolerance as you see fit

# Create GAMSPy container
m = Container()

# Sets
i = Set(m, name="i", records=assets)
ii = Alias(m, name="ii", alias_with=i)

# Parameters
Q = Parameter(m, name="Q", domain=[i, ii],
              records=[(assets[r], assets[c], cov[r, c]) for r in range(n_assets) for c in range(n_assets)])

mu = Parameter(m, name="mu", domain=[i],
               records=[(assets[k], mu_array[k]) for k in range(n_assets)])

x_prev = Parameter(m, name="x_prev", domain=[i],
                   records=[(assets[k], x_prev_array[k]) for k in range(n_assets)])

c = Parameter(m, name="c", domain=[i],
              records=[(assets[k], c_array[k]) for k in range(n_assets)])

# Scalar parameter for max risk (variance)
sigma_max = sigma_max_sq  # just use a plain Python scalar

# Variables
x = Variable(m, name="x", domain=[i], type="Free")  # portfolio weights (can be zero or positive)
x.lo[i] = 0  # no short selling

y = Variable(m, name="y", domain=[i], type="Binary")  # selection binary variables

# Constraints
budget = Equation(m, name="budget")
budget[...] = Sum(i, x[i]) == 1

link = Equation(m, name="link", domain=[i])
link[i] = x[i] <= y[i]  # link between selection and investment

cardinality = Equation(m, name="cardinality")
cardinality[...] = Sum(i, y[i]) <= 5  # max 5 assets selected

risk = Equation(m, name="risk")
risk[...] = Sum([i, ii], Q[i, ii] * x[i] * x[ii]) <= sigma_max

# Objective: maximize expected return minus transaction cost (transaction cost is quadratic)
transaction_cost = Sum(i, c[i] * (x[i] - x_prev[i]) * (x[i] - x_prev[i]))
objective_expr = Sum(i, mu[i] * x[i]) - transaction_cost

# Model
model = Model(m, name="CCPOPwTCaRL5_538",
              equations=m.getEquations(),
              problem="MIQCP",
              sense=Sense.MAX,
              objective=objective_expr)

# Solve
import sys
#model.solve(output=sys.stdout, solver="convert")

#model.solve(output=sys.stdout, solver="SBB")
model.solve(output=sys.stdout, solver="SHOT", solver_options={"Dual.MIP.Solver": 2})